<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex10.2-solid-oxide-cell/Ex10.2_01_button_cell_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

### What fitting synthetic data does and does not show

This notebook generates a polarisation curve from the model, adds noise, and
fits the same model back to it. That is worth doing once, because it verifies
the fitting machinery and shows how noise propagates into the parameters.

**It does not validate the model.** The data was produced by the very equations
being fitted, so agreement is guaranteed by construction and says nothing about
whether those equations describe a real cell. This is sometimes called an
inverse crime, and it is a common way of appearing to validate something that
has not been validated.

The real test is the one in the verification notebook: reproduce a **published**
polarisation curve, measured on a real cell at a stated temperature and gas
composition, with parameters that were not fitted to it.


# Ex_10.2 · Notebook 01 — the 0-D button cell

**Paired with L10.2 · Solid oxide cells**

Fit the model to a polarisation curve, then verify it against the four checks
of L10.2 slide 9.

Published button-cell fits report activation energies of order 100 kJ/mol for
the fuel electrode and 66 kJ/mol for the oxygen electrode, with exchange
current densities of order 10⁻² A/cm² — use those as sanity bounds on anything
you fit.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex10.2-solid-oxide-cell/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## TODO 1 — synthesise a "measurement" and fit to it

Generate a curve with known parameters, add noise, then recover them. If you
cannot recover parameters you generated yourself, the fit is not trustworthy
on real data either.

In [ ]:
truth = pb.SOCParams("SOEC", 1073.15, 0.10, 0.90, ASR_0=0.31, i_L=2.2, i_0=0.42)
i_meas = np.linspace(-1.0, 1.4, 40)
rng = np.random.default_rng(3)
V_meas = pb.cell_voltage(i_meas, truth) + 0.004 * rng.standard_normal(i_meas.shape)

def sse(ASR_0, i_L, i_0):
    p_ = pb.SOCParams("SOEC", truth.T, truth.p_H2, truth.p_H2O,
                      ASR_0=ASR_0, i_L=i_L, i_0=i_0)
    return float(((pb.cell_voltage(i_meas, p_) - V_meas) ** 2).sum())

# TODO 1 --- fit the three parameters to the measured curve ---------------------------------------------
# Two `...` to replace:
#   line 1  ->  sse(x[0], x[1], x[2])                  the objective, unpacked from one vector
#   line 2  ->  [0.25, 2.5, 0.5]                        the starting guess: the catalogue defaults
from scipy.optimize import minimize
res = minimize(lambda x: ...,                     # <- sse(x[0], x[1], x[2])
               x0=...,                            # <- [0.25, 2.5, 0.5]
               method="Nelder-Mead", options={"xatol": 1e-5, "fatol": 1e-9, "maxiter": 4000})
fit = {"ASR_0": float(res.x[0]), "i_L": float(res.x[1]), "i_0": float(res.x[2])}
fitted = pb.SOCParams("SOEC", truth.T, truth.p_H2, truth.p_H2O, **fit)
print(f"  truth  ASR_0 {truth.ASR_0:.3f}  i_L {truth.i_L:.2f}  i_0 {truth.i_0:.2f}")
print(f"  fit    ASR_0 {fit['ASR_0']:.3f}  i_L {fit['i_L']:.2f}  i_0 {fit['i_0']:.2f}")
# ------------------------------------------------------------------------------

## TODO 2 — the four verification checks

1. OCV at zero current equals the Nernst potential.
2. Heat generation crosses zero at $V_{tn}$.
3. The ohmic slope changes between two temperatures by the Arrhenius factor.
4. The curve reproduces the measured points within the noise level.

In [ ]:
# TODO 2 --- the four verification checks ---------------------------------------------------------------
# Four `...` to replace, one per check:
#   line 1  ->  pb.nernst(truth)                                              the OCV must equal the Nernst potential
#   line 2  ->  pb.thermoneutral_voltage(truth.T)                             heat crosses zero where V = V_tn
#   line 3  ->  np.exp(truth.E_act_ASR / pb.R_GAS * (1/T1 - 1/T2))            the Arrhenius factor between two temperatures
#   line 4  ->  0.004                                                          the noise you added, in volts
# check 1: OCV at zero current
check("OCV at i = 0", pb.cell_voltage(0.0, truth), ..., tol=1e-9)         # <- pb.nernst(truth)

# check 2: heat generation crosses zero at the thermoneutral voltage
i_grid = np.linspace(0.0, 1.4, 20001)
i_tn = i_grid[np.argmin(np.abs(pb.cell_voltage(i_grid, truth) - ...))]    # <- pb.thermoneutral_voltage(truth.T)
check("heat at V_tn", float(pb.heat_generation(i_tn, truth)), 0.0, tol=1e-4)

# check 3: the ohmic slope changes between two temperatures by the Arrhenius factor
T1, T2 = 973.15, 1073.15
ratio = float(pb.eta_ohmic(1.0, truth, T1) / pb.eta_ohmic(1.0, truth, T2))
check("Arrhenius ratio of ASR", ratio, ..., tol=1e-9)                      # <- np.exp(truth.E_act_ASR / pb.R_GAS * (1/T1 - 1/T2))

# check 4: the fitted curve reproduces the measured points within the noise
rel = relative_l2(pb.cell_voltage(i_meas, fitted), V_meas)
mx  = max_abs_error(pb.cell_voltage(i_meas, fitted), V_meas)
print(f"  fitted curve: relative L2 {rel:.2e}, worst point {mx:.4f} V")
assert mx < 3 * ..., "worst point should sit within about three noise standard deviations"   # <- 0.004
print("  PASS  fitted curve within the noise")
# ------------------------------------------------------------------------------

In [ ]:
import pickle

os.makedirs("Ex10.2_outputs", exist_ok=True)
# with open(os.path.join("Ex10.2_outputs", "ex102_fit.pkl"), "wb") as f:
#     pickle.dump({"par": truth, "OCV": float(pb.nernst(truth)), ...}, f)
#
# Notebook 05 reads every .pkl it finds in Ex10.2_outputs and puts one row in
# the report per case, so store a dict with a "par" key holding the SOCParams.